# Step 1 Phase A-10：W₂ 推定量・orientation-cluster m 感度・global 較正経路 **v1.2.5**
2026-09-12。**v1.2.4 監査（ChatGPT）**：(1) cross-selection helper を `compare_selection_paths(d_sensitivity, d_primary)` に改名し evidence label を `*_sensitivity／*_primary` に；calibration の自己比較（f64 vs f64）を **cal32 vs cal64** に修正，m run の引数順を修正，preflight に alias assert（`cal is cal64`・`cal32 is not cal64`）と self-test に「自己比較は異常を検出できない」negative case を追加；(2) `r64/hM64/hI64/f64sel` → `r32/hM32/hI32/f32sel`；(3) W₂ の selection 感度を **exact subsample bound（修正案 B）** に：OT に投入した cluster subset ごとの paired RMS ε から observed の pair 差上界と null replicate ごとの W₂_max 摂動上界（`delta_q99_bound`）を計算し，**decision margin gate** `|W₂_max − q99| > observed_bound + delta_q99_bound`（official required）；full-pool RMS は診断に降格；(4) provenance／コメントを f64 primary に統一（`scan` の default 削除・W₂ 冒頭・positive control 0.35）；(5) **chunk を A8b CPU CSV の dtype 別 winner に binding**（f64 primary=2000・f32 sensitivity=20000；`G_a8b_chunk_binding`）；(6) A10c の f32 感度は「primary pseudo threshold 固定の conditional sensitivity」と明記。

（v1.2.4）2026-09-12。**v1.2.3 監査（ChatGPT）の第一推奨を採用：ℓ2–4 の selection を float64 primary に**（A8b freeze の float32 は性能 benchmark の推奨として保持し，A10 の広い stress evidence——非対蹠 near-tie flip・antipode flip での T₂ 差 ≤5e-3——により **ℓ2–4 の dtype 推奨のみ supersede**；S2 は float32 のまま）。float32 は A8b 登録 path の **sensitivity** として全 component で併走：m run（Q 直接感度 |logQ₆₄−logQ₃₂| と near-tie／Event B 一致），calibration（一致 gate），A10c（float32 hit table による support 配列と false-support 頻度の差を保存），**W₂（paired whitened 出力の coupling 上界 ε_j=√mean‖z₆₄−z₃₂‖² と三角不等式による pair 差上界を null q99 に対して gate**）。旧 `G_f32_selection_plane_equiv_probe` を helper の 5 ケース self-test に置換；相対差分母を tiny で保護；generation call の exact inventory；非対蹠 flip の plane-folded 角距離を evidence に保存；m run の full AX（両 path・int16）を保存。

（v1.2.3）2026-09-12。**v1.2.2 監査（ChatGPT）**：(1) cross-selection policy を **official 前に固定**。サンドボックスの v1.2.3 smoke で，plane-equivalent な antipode flip でも T₂ が最大 7.5e-4 相対で変わる標本（B⁻ 行が同一でない antipodal representative）と，**非対蹠の plane への flip**（2 平面の S⁺ が float32 分解能内の near tie）が各 ~5e-5/標本で観測された。従って「plane 同一＋連続出力 1e-5」は float32 production path では成立しない。登録する policy（**production float32 が authoritative，float64 selection は sensitivity path**）：共通 helper `compare_selection_paths` で (a) flip は **selected S⁺ の相対差 < `TOL_NEAR_TIE = 1e-6`（float32 分解能）の near tie でのみ起きる**こと，(b) **Event B mismatch 率 ≤ `TOL_EVENTB_MISMATCH = 1e-5`**（P(E_B)≈4e-3 に対し 0.25% 未満の相対影響）を required（`G_cal_f32_f64_selection_consistency`・`G_m_f32_f64_selection_consistency_all_runs`），antipodal/非対蹠 flip 数・T₁/T₂ 差・flip 行の evidence は保存。A8b の 1e-6 評価許容は書き換えない；flip 行の evidence（index・AX32/AX64・antipode 関係・T₁/T₂ 両 path・Event B 両 path）を compact NPZ に保存；axis は raw oriented representative と plane-folded の双方を保存し T₁/T₂ は raw representative の float64 B± 行で評価すると明記。(2) 乱数 stream を purpose namespace registry（`GEN_NS`）で管理し全 generation call を記録，`G_generation_stream_keys_unique` を required に（calibration と pseudo の衝突を解消）。

（v1.2.2）2026-09-12。**v1.2.1 監査（ChatGPT）**：(1) `scan()` に A8b child と同じ非有限 fail-fast（selected score・T₁・T₂）；(2) positive control の full finite／D>0 inventory と observed positive control の finite・分母>0；(3) A10b の二重 diff_ci を削除；(4) W₂ checkpoint binding に IND/CRN/TwI 配列 SHA・platform・THREADS_LIVE を追加；(5) m run の等方側 f32/f64 sensitivity（T₁/T₂ 最大差）と 5 seed の equivalence verdict を保存；(6) required と diagnostic の区別を provenance に明示。

（v1.2.1）2026-09-12。**v1.2 監査（ChatGPT）§11**：m ごとの finite/precision/consistency を分けた policy gate（m=10 fallback が塞がれない）；W₂ checkpoint binding に notebook source・環境・asset SHA・whitening・S_POS SHA を含め payload も保存；negative control の valid fraction・finite inventory gate；W₂ 全ペア値の finite assert/gate（`W2_max` だけでなく）；warning を n_sub 別に集計；CKPT の exact filename inventory；5 bootstrap seed の分布を保存；Δ_m=log 1.10 の科学的位置づけを provenance に明文化；MC p は finite-pool exceedance と呼ぶ；返送パスを v1.2.1 に。

（v1.2）2026-09-12。**v1.1 監査（ChatGPT 2026-09-11）§15 の 15 項目を反映**：A8b child と同一の float32 丸め順（x→float32→積→GEMM）と float64 評価（einsum）を再実装し，A8b child のコードを逐語的に inline 再現した regression gate；m=10 fallback が通る precision-policy gate；**practical equivalence margin Δ_m = log 1.10 を事前登録**（差 CI が ±Δ_m 内で m=100，そうでなければ m=10，unresolved は fail-fast）；較正 bootstrap で分子 0 を 0 として保持・valid fraction gate・negative control は独立 bootstrap・positive control は D 分岐も検査・main pathway の finite/inventory gate；3 位置すべての matched-power/sqrt gate と全ペア非等価；W₂ の **primary を独立 (R,z) stream**（null と同じ sampling mechanism）とし CRN 版は conservative diagnostic に降格，observed/pair/seed/null-length/stability gate（3 seed・n_sub 2000/5000 の判定一致・W₂_max 相対 spread ≤ 0.25 を事前登録），component 名を `W2_PATHWAY_VALID` に；A9 provenance の M21/LM/real-basis/quadrature hash への binding・共分散 real transform gate；official では live notebook source と BLAS thread 数を hard gate；NPZ も atomic。v1.1 の説明は以下に残す。

（v1.1）**v1.0 監査（ChatGPT 2026-09-11）の全項目を反映**。v1.0 は A5 凍結前の中央帯 event と float64 selection・chunk 10000 で作られており（新チャットで設計書から起こしたことによる回帰），その結果（Q≈1.08–1.11・m=100・FWFSR 0/200）は **superseded**。
v1.1：(1) primary event を **A5 凍結の Event B = {T₁ ≤ T₁,obs ∧ T₂ ≤ T₂,obs}** に置換（pseudo は 2D 閾値）；(2) **A8b 凍結 production path**（`l24_feature231`・float32 selection・float64 evaluation・chunk 20000・BLAS threads 2・plane-folded axis）を primary，float64 selection を sensitivity 併記；(3) m 判定を 4 run・fail-closed（両 rep ペアと pooled の差 CI が 0 を含み全 run 精度 gate → m=100／m=100 不適格かつ m=10 両 rep 適格 → m=10／それ以外 unresolved），5 gate 追加，4 run の cluster 集計を保存；(4) 較正経路を 2D pseudo 閾値に修正し **one-point pathway prototype** として名称・status を分離，negative／synthetic boosted positive control・brute-force 一致を hard gate；(5) W₂ を **3 位置 max-pairwise**（A11 cache の非等価 x₀ 3 点を mock として使用）に拡張，null は replicate ごとに disjoint cluster block・3 標本 max，quantile 法固定・MC p・B／n_sub／seed 感度；(6) A9/A8/A5/A11 freeze への SHA binding・module purge＋live file SHA・official では版 hard gate・POT pin・BLAS threads 固定・CVEC ℓ-block 一定・Mx basis vs M21・A5 null file SHA・出力 SHA・atomic write・component status → `A10_VALID`・final assert。
cluster ESS 要件は superseded（precision gate = positive cluster 数＋CI 相対半幅＋5-seed 幅 CV）と明記。Phase A 調査ノート：数値は rules v1.0 draft の設計根拠。


In [ ]:
# ---- A10a: mode / environment / pinned repo / module purge + live SHA / asset binding / engine / calibration sample / A5 cross-check ----
import os, sys, json, hashlib, subprocess, platform, time, warnings, importlib
A10_MODE = globals().get('A10_MODE', 'official'); IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
MT_COMMIT = '1bdd9ea8a00891d6dc3622331f6dad5b86c16c89'                                              # A8 freeze commit (A5/A9/A11/A8 freezes, t1_engine, t2b2_bridge, t2b2_run, Step 0)
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False); BASE = '/content/drive/MyDrive/mirror_topology'; WORK = '/content/a10_work'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pot==0.9.7.post1', 'threadpoolctl'], check=True)
else: BASE = os.environ.get('A10_BASE', '/tmp/a10w/base'); WORK = os.environ.get('A10_WORK', '/tmp/a10w')
os.makedirs(WORK, exist_ok=True); MT = os.path.join(WORK, 'mt_a10'); GIT_CALLS = []
def git(*a, check=True): r = subprocess.run(['git', '-C', MT, *a], capture_output=True, text=True); GIT_CALLS.append(['git', *a, r.returncode]); assert (r.returncode == 0) or not check, (a, r.stderr[:300]); return r.stdout.strip()
if not os.path.exists(os.path.join(MT, '.git')): subprocess.run(['git', 'clone', '-q', 'https://github.com/tsujikeita/mirror-topology.git', MT], check=True)
git('fetch', '-q', 'origin', MT_COMMIT, check=False); git('checkout', '-q', '--force', MT_COMMIT); git('clean', '-fdxq')
GATES, DIAG, REC, STATUS = {}, {}, {}, {}
GATES['G_repo_commit'] = (git('rev-parse', 'HEAD') == MT_COMMIT); GATES['G_repo_origin'] = ('tsujikeita/mirror-topology' in git('remote', 'get-url', 'origin')); GATES['G_repo_clean'] = (git('status', '--porcelain', '--untracked-files=all') == '')
OUT = os.path.join(BASE, 'runs_step1_phaseA', f'a10_v1.2.5_{A10_MODE}'); CKPT = os.path.join(OUT, 'checkpoints'); os.makedirs(CKPT, exist_ok=True)
def sha(p, block=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''): h.update(b)
    return h.hexdigest()
def _jsonable(o):
    import numpy as _np
    if isinstance(o, (_np.bool_,)): return bool(o)
    if isinstance(o, (_np.integer,)): return int(o)
    if isinstance(o, (_np.floating,)): return float(o)
    if isinstance(o, _np.ndarray): return o.tolist()
    raise TypeError(f'not JSON serializable: {type(o).__name__}')
def atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as fh: json.dump(obj, fh, indent=1, ensure_ascii=False, default=_jsonable); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)
def atomic_npz(path, **arrays):
    tmp = path + '.tmp'
    with open(tmp, 'wb') as fh: np.savez_compressed(fh, **arrays); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)
# module purge + live file SHA (python reuses already-imported modules)
for m in ('t1_engine', 't2b2_bridge', 't2b2_run'): sys.modules.pop(m, None)
sys.path.insert(0, MT); import t1_engine as t1, t2b2_bridge as br, t2b2_run as tr
EXP_MOD = dict(t1_engine='87bf8424073af021264b12fe312ab5255b71008bdd5fe874d164d48daf034dc8', t2b2_bridge='45107d1608d50816712f1aa452d9fa39af4adc9ec035fbe9279b264760d65872', t2b2_run='03c80f2136a8ff7ffb1077749895811ef95dd9d779c7996891d89e75545ff8db')
GATES['G_live_modules'] = all(os.path.realpath(mod.__file__).startswith(os.path.realpath(MT)) and sha(mod.__file__) == EXP_MOD[n] for n, mod in (('t1_engine', t1), ('t2b2_bridge', br), ('t2b2_run', tr)))
import numpy as np, scipy, healpy as hp, pandas as pd, ot, threadpoolctl
from scipy.spatial.transform import Rotation
from scipy.special import sph_harm_y
warnings.filterwarnings('ignore')
VERS = dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__, healpy=hp.__version__, pandas=pd.__version__, pot=ot.__version__)
EXPECTED_VERS = dict(python='3.13.15', numpy='2.1.3', scipy='1.16.3', healpy='1.20.0', pot='0.9.7.post1')      # Colab official environment (A8/A11 official) + pinned POT
VERS_MISMATCH = {k: (VERS[k], v) for k, v in EXPECTED_VERS.items() if VERS[k] != v}
GATES['G_env_versions'] = (len(VERS_MISMATCH) == 0) if A10_MODE == 'official' else True             # hard gate in official; smoke records only
THREADS = 2; threadpoolctl.threadpool_limits(THREADS); THREADS_LIVE = [dict(api=i['user_api'], lib=i['internal_api'], n=i['num_threads']) for i in threadpoolctl.threadpool_info()]
GATES['G_threads_live_registered'] = (all(i['n'] == THREADS for i in THREADS_LIVE if i['api'] == 'blas') and any(i['api'] == 'blas' for i in THREADS_LIVE)) if A10_MODE == 'official' else True
# notebook identity (committed copy vs live is recorded; A10 is a Phase A investigation notebook, so head_copy may be absent)
NB_BASENAME = 'MirrorTopology_Step1_A10_v1.2.5.ipynb'
git('fetch', '-q', 'origin', 'main', check=False)
try: NB_HEAD = tr.source_only_sha(subprocess.run(['git', '-C', MT, 'show', f'origin/main:{NB_BASENAME}'], capture_output=True, check=True).stdout)
except Exception: NB_HEAD = None
NB_LIVE = tr.live_notebook_source_sha(); GATES['G_notebook_live_source'] = (isinstance(NB_LIVE, str) and NB_HEAD is not None and NB_LIVE == NB_HEAD) if A10_MODE == 'official' else True   # official: the committed notebook must be the one executing
# ---- frozen asset binding ----
P = lambda *a: os.path.join(MT, *a)
ASSETS = dict(bstack=(P('results/step1_phaseA/A5_freeze/s1_Bstack_l2_4_N16_common_v1.npz'), 'ec2d3eb501c3e00af85a505d95d7141fddb4ea23ab1da971906a5c21f80eef5f'),
              a5_null=(P('results/step1_phaseA/A5_freeze/a5_null_selection.npz'), 'd0de2cf6643ff19e134567616f243298b2cc83dd081032949984c1c6abfa2b9e'),
              a5_prov=(P('results/step1_phaseA/A5_freeze/a5_provenance.json'), '33765464cd4f4bea67e78fddb769e92ccbea38e68f6ec74778a203309971ba80'),
              a9_prov=(P('results/step1_phaseA/A9_freeze/a9_v1.2.1/a9_provenance.json'), '09c03219f43ce69368c238ab61567c3e804a45302386485820b45534539d16ef'),
              a9_manifest=(P('results/step1_phaseA/A9_freeze/freeze_manifest.json'), '4b454a157b0bdbed4a6c8856b750bb0a8951f042c56a596a4971a1bc092e2c9a'),
              a9_script=(P('results/step1_phaseA/A9_freeze/s1_phaseA9_v1.2.1.py'), '2905c036a04a04e33672a07ba464d95ae6a669c8ea7b918f17a6813e864dde2c'),
              a8_manifest=(P('results/step1_phaseA/A8_freeze/freeze_manifest.json'), '3384c9c5994a8c656ca1c868ff60156ad7e627a038bfeffe320972a6fc72e031'),
              a8b_prov=(P('results/step1_phaseA/A8_freeze/a8b/official/a8b_provenance.json'), 'a5314043e025a00ffe7ca6d89985bb1b6064e398bc9ef911de7348d601c85d6a'),
              a11_manifest=(P('results/step1_phaseA/A11_freeze/freeze_manifest.json'), '6000d7b7049dc5232f3e87daab2d4e56cb069cd15145f252ba4a548bf94df981'),
              cov_P1=(P('results/step1_phaseA/A11_freeze/official/cov_cache/cov_E7_99337bfd75deea5fe.npy'), '27e2bb589722526a40ddeca1b738f6c58edb1c8efd39672b0f7080d8de133308'),
              cov_P2=(P('results/step1_phaseA/A11_freeze/official/cov_cache/cov_E7_ebc72653fa1a10e7a.npy'), '727772a10af23118f4cb4c725af369578c1d3ac9d52d0cb641e4fc6399c29219'),
              cov_P3=(P('results/step1_phaseA/A11_freeze/official/cov_cache/cov_E7_b1862f0a867b6b3de.npy'), 'dbec1b921c3d74847095d364c04222f795c59ce4eb86686817e70d7d861d53fe'),
              step0_npz=(P('docs/step0_frozen_Bpm_v1.npz'), None), step0_csv=(P('results/step0_v0.7/step0_official_v0_7.csv'), None))
ASSET_SHA = {k: sha(p) for k, (p, e) in ASSETS.items()}; GATES['G_asset_file_sha'] = all(ASSET_SHA[k] == e for k, (p, e) in ASSETS.items() if e)
a9p = json.load(open(ASSETS['a9_prov'][0])); GATES['G_a9_binding'] = (a9p['OFFICIAL'] is True and all(a9p['gates'].values()) and len(a9p['gates']) == 30); A9B, A9Q = a9p['records']['bridge'], a9p['records']['quadrature']
a8p = json.load(open(ASSETS['a8b_prov'][0])); L24 = a8p['ROUTE_DECISION']['l24']
A8B_L24 = dict(route='l24_feature231', selection_dtype='float32', evaluation_dtype='float64', sample_chunk=20000, threads=2)                   # A8b frozen benchmark recommendation (performance)
a8csv = pd.read_csv(P('results/step1_phaseA/A8_freeze/a8b/official/a8b_benchmark_cpu.csv')); _w = a8csv[(a8csv.route == 'l24_feature231') & (a8csv.stage == 'winner') & (a8csv.status == 'ok')]
A8B_WINNER = {r.selection_dtype: dict(chunk=int(r.chunk), N=int(r.N), production_s_median=float(r.production_s_median), peak_increment_GB=float(r.peak_increment_GB)) for r in _w.itertuples()}
CHUNK_BY_SEL = dict(float64=2000, float32=20000)                                                                                       # per-dtype chunk = A8b official CPU winner for l24_feature231
GATES['G_a8b_chunk_binding'] = (A8B_WINNER.get('float64', {}).get('chunk') == 2000 and A8B_WINNER.get('float32', {}).get('chunk') == 20000 and A8B_WINNER['float64']['N'] == A8B_WINNER['float32']['N'] == 1_000_000)
PROD = dict(route='l24_feature231', selection_dtype='float64', evaluation_dtype='float64', sample_chunk=CHUNK_BY_SEL['float64'], sample_chunk_sensitivity=CHUNK_BY_SEL['float32'], threads=2, primary='float64', sensitivity='float32', a8b_winners=A8B_WINNER,
            supersede_note='A10 supersedes the l2-4 selection dtype only (float32 -> float64, cost ~1.3 min/1e6) on stress evidence: non-antipodal near-tie flips and antipodal flips with T2 rel diff up to 5e-3; A8b benchmark stays valid; S2 keeps float32')
GATES['G_a8b_production_spec_bound'] = (a8p['status'] == 'BENCHMARK_VALID' and all(L24[k] == v for k, v in A8B_L24.items() if k != 'threads') and a8p['registered']['threads'] == 2)
a5p = json.load(open(ASSETS['a5_prov'][0])); GATES['G_a5_binding'] = (a5p.get('status') in ('OFFICIAL', 'VALID', 'PASS', None) and all(a5p['gates'].values()))
zB = np.load(ASSETS['bstack'][0]); Bp, Bm = np.asarray(zB['Bp_stack'], np.float64), np.asarray(zB['Bm_stack'], np.float64)
GATES['G_bstack_array_sha'] = (hashlib.sha256(np.ascontiguousarray(Bp).tobytes() + np.ascontiguousarray(Bm).tobytes()).hexdigest() == 'eb51414885785b77e9d3f7fbb25e1c9396f52e19c053d58113d50e206353a93f')
z0 = np.load(ASSETS['step0_npz'][0], allow_pickle=True); CVEC = np.asarray(z0['CVEC'], np.float64); RB = br.real_basis_lm(); LM = br.lm_full(); M21 = br.M_matrix()[0]
GATES['G_cvec_sha'] = (hashlib.sha256(np.ascontiguousarray(CVEC).tobytes()).hexdigest() == '17d85b41ee0665d88418ebf8dee794d0da0a6f219053e983ec9b0501d763c8c6')
LBLK = [(slice(0, 5), 2), (slice(5, 12), 3), (slice(12, 21), 4)]; GATES['G_cvec_lblock_constant'] = all(np.ptp(CVEC[b]) == 0 for b, l in LBLK)
GATES['G_basis_order'] = ([tuple(b) for b in z0['basis_lm']] == [(int(l), int(m), cs) for (l, m, cs) in RB] and [(int(l), int(m)) for (l, m) in LM] == [(l, m) for l in (2, 3, 4) for m in range(-l, l + 1)])
asha = lambda a: hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
GATES['G_a9_basis_hashes'] = (asha(M21) == A9B['M21_sha256'] and hashlib.sha256(json.dumps(LM).encode()).hexdigest() == A9B['LM_sha256'] and hashlib.sha256(json.dumps(RB).encode()).hexdigest() == A9B['real_basis_sha256'])
s0 = pd.read_csv(ASSETS['step0_csv'][0]); r0 = s0[s0['map'] == 'PR3_Commander']; T1o, T2o = 39.67178834527284, 259.3375006282747
GATES['G_step0_obs_bound'] = (len(r0) == 1 and float(r0.iloc[0]['Splus']) == T1o and float(r0.iloc[0]['Sminus']) == T2o)
def event_B(T1, T2, t1=T1o, t2=T2o): return (T1 <= t1) & (T2 <= t2)                                   # A5-frozen primary event
# ---- covariance systems: PR3-power-matched (primary), CT-native diagnostics recorded ----
def load_C(key):
    Mx, Cr, meta = t1.load_cov_full(ASSETS[key][0], 4); man = json.load(open(ASSETS[key][0] + '.manifest.json'))
    assert meta['cov_array_sha256'] == man['cov_array_sha256'] and man['manifest']['topology'] == 'E7' and man['manifest']['params'] == dict(LAx=1.0, LAy=0.3, L1y=1.0, L2x=0.0, L2z=1.0)
    Csym = Mx; Cr_chk = (M21 @ Csym @ M21.conj().T).real if Csym.shape == (21, 21) else None                                          # real-basis transform must be the bridge M21: C_real == Re(M21 Csym M21^H) (t2b2_bridge.to_real)
    return Cr, man['manifest']['x0'], meta, (Cr_chk is not None and np.allclose(Cr_chk, Cr, rtol=1e-12, atol=1e-12 * np.abs(Cr).max()) and meta['real_transform']['ok'])
C_CT = {}; X0 = {}; CTM = {}; CB = {}
for k in ('cov_P1', 'cov_P2', 'cov_P3'): C_CT[k], X0[k], CTM[k], CB[k] = load_C(k)
GATES['G_cov_basis_matches_M21'] = all(CB.values())
GATES['G_cov_positions_distinct'] = (len({tuple(np.round(v, 6)) for v in X0.values()}) == 3 and all(np.linalg.norm(C_CT[a] - C_CT[b]) / np.linalg.norm(C_CT[a]) > 1e-6 for a in C_CT for b in C_CT if a < b))
c_pr3 = CVEC[[0, 5, 12]]
def matched(C):
    c_ct = np.array([np.trace(C[b, b]) / (2 * l + 1) for b, l in LBLK]); Dm = np.diag(np.concatenate([np.repeat(np.sqrt(c_pr3[i] / c_ct[i]), 2 * l + 1) for i, (b, l) in enumerate(LBLK)])); return Dm @ C @ Dm, c_ct
C_MP = {k: matched(C_CT[k]) for k in C_CT}; C_M, c_ct1 = C_MP['cov_P1']; C_ISO = np.diag(CVEC)
GATES['G_matched_power'] = all(np.allclose([np.trace(C_MP[k][0][b, b]) / (2 * l + 1) for b, l in LBLK], c_pr3, rtol=1e-12) for k in C_MP)
def psqrt(C):
    w, V = np.linalg.eigh(C); wc = np.where(w < 1e-12 * w.max(), 0.0, w); S = V @ np.diag(np.sqrt(wc)) @ V.T
    return S, dict(lambda_min=float(w.min()), clip=int(np.sum(w < 1e-12 * w.max())), sym=float(np.linalg.norm(S - S.T) / np.linalg.norm(S)), recon=float(np.linalg.norm(S @ S.T - C) / np.linalg.norm(C)))
SQ = {k: psqrt(C_MP[k][0]) for k in C_MP}; S_M, iM = SQ['cov_P1']; S_I, iI = psqrt(C_ISO); S_POS = [SQ[k][0] for k in ('cov_P1', 'cov_P2', 'cov_P3')]
GATES['G_sqrt_hard'] = all(v[1]['sym'] < 1e-12 and v[1]['recon'] < 1e-10 and v[1]['clip'] == 0 and v[1]['lambda_min'] > 0 for v in SQ.values()) and iI['sym'] < 1e-12 and iI['recon'] < 1e-10
REC['covariance'] = dict(primary_system='PR3-power-matched (morphology-only covariance experiment)', model_point='E7_b1_A x0(1)=(0.31,0.21,0.42)', c_l_CT={k: C_MP[k][1].tolist() for k in C_MP}, c_l_PR3=c_pr3.tolist(), sqrt={k: SQ[k][1] for k in SQ}, sqrt_iso=iI, ct_intake={k: CTM[k]['eig'] for k in CTM}, positions={k: X0[k] for k in X0}, real_transform={k: CTM[k]['real_transform'] for k in CTM})
# ---- D(R): A9/A11 quadrature construction, bound to A9 and regression-tested by direct geometry + known z-rotation ----
LMAX = 4; _NT = _NP = 2 * LMAX + 2; _xg, _wg = np.polynomial.legendre.leggauss(_NT); _th = np.arccos(_xg); _ph = 2 * np.pi * np.arange(_NP) / _NP
TH, PH = np.meshgrid(_th, _ph, indexing='ij'); WQ = (np.repeat(_wg[:, None], _NP, axis=1) * (2 * np.pi / _NP)).ravel()
DIRS = np.column_stack([np.sin(TH).ravel() * np.cos(PH).ravel(), np.sin(TH).ravel() * np.sin(PH).ravel(), np.cos(TH).ravel()])
def Yc_at(dirs):
    th, ph = hp.vec2ang(dirs); return np.array([sph_harm_y(l, m, th, ph) for (l, m) in LM])
def Ymat(dirs): return (M21.conj() @ Yc_at(dirs)).real.T
YQ = Ymat(DIRS); YQW = (YQ * WQ[:, None]).T; GATES['G_quadrature_orthonormal'] = bool(np.abs(YQW @ YQ - np.eye(21)).max() < 1e-12)
DIAG['quadrature_sha256'] = dict(dirs=asha(DIRS), weights=asha(WQ), M21=asha(M21)); GATES['G_a9_quadrature_hashes'] = (asha(DIRS) == A9Q['nodes_sha256'] and asha(WQ) == A9Q['weights_sha256'] and _NT == A9Q['n_theta'] and _NP == A9Q['n_phi'])
def D_of_R(Rm): return YQW @ Ymat(DIRS @ Rm)
def D_batch(Rs):
    Pp = np.einsum('qj,kji->kqi', DIRS, Rs); th, ph = hp.vec2ang(Pp.reshape(-1, 3)); Y = np.array([sph_harm_y(l, m, th, ph) for (l, m) in LM]).reshape(21, len(Rs), -1)
    Yr = np.einsum('ab,bkq->kqa', M21.conj(), Y).real; return np.einsum('aq,kqb->kab', YQW, Yr)
_rg = np.random.default_rng(20260912); Rt = Rotation.random(num=8, rng=_rg).as_matrix(); Db = D_batch(Rt); dirs_t = _rg.standard_normal((200, 3)); dirs_t /= np.linalg.norm(dirs_t, axis=1, keepdims=True)
GATES['G_D_batch_matches_single'] = bool(max(np.abs(Db[k] - D_of_R(Rt[k])).max() for k in range(8)) < 1e-12)
GATES['G_D_orthogonal'] = bool(np.abs(np.einsum('kab,kac->kbc', Db, Db) - np.eye(21)).max() < 1e-10); GATES['G_D_homomorphism'] = bool(np.abs(D_of_R(Rt[0] @ Rt[1]) - Db[0] @ Db[1]).max() < 1e-10)
xg = _rg.standard_normal(21); ag = M21.conj().T @ xg
GATES['G_D_direct_geometry'] = bool(max(np.abs((Yc_at(dirs_t).T @ (M21.conj().T @ (Db[k] @ xg))).real - (Yc_at(dirs_t @ Rt[k]).T @ ag).real).max() / np.abs((Yc_at(dirs_t @ Rt[k]).T @ ag).real).max() for k in range(8)) < 1e-10)
al = 0.7; Rz = np.array([[np.cos(al), -np.sin(al), 0], [np.sin(al), np.cos(al), 0], [0, 0, 1]]); Dz = D_of_R(Rz); okz = True
for i, (l, m, cs) in enumerate(RB):                                                                    # known z-rotation: (c,s) pairs of the same (l,m) mix by cos/sin of m*alpha, m=0 fixed
    if m == 0: okz &= abs(Dz[i, i] - 1) < 1e-10
    elif cs == 'c': j = i + 1; blk = Dz[np.ix_([i, j], [i, j])]; okz &= (np.allclose(blk, [[np.cos(m * al), -np.sin(m * al)], [np.sin(m * al), np.cos(m * al)]], atol=1e-10) or np.allclose(blk, [[np.cos(m * al), np.sin(m * al)], [-np.sin(m * al), np.cos(m * al)]], atol=1e-10))
GATES['G_D_known_z_rotation'] = bool(okz and np.abs(Dz - np.diag(np.diag(Dz))).sum() - sum(abs(Dz[i, i + 1]) + abs(Dz[i + 1, i]) for i, (l, m, cs) in enumerate(RB) if m != 0 and cs == 'c') < 1e-9)
# ---- production scan: float64 selection PRIMARY for l2-4 (A10 supersedes the A8b dtype recommendation; chunk = A8b float64 winner), float32 selection = A8b registered SENSITIVITY path (chunk = A8b float32 winner); float64 evaluation at the selected axis; plane-folded axis id ----
iu = np.triu_indices(21); off = (iu[0] != iu[1])
def fvec(B): v = B[iu].copy(); v[off] *= 2.0; return v                                                   # A8b child: feature vector of an axis matrix (off-diagonal doubled, in the selection dtype)
Bp32 = Bp.astype(np.float32); Fp32 = np.array([fvec(Bp32[a]) for a in range(3072)], dtype=np.float32); Fp64 = np.array([fvec(Bp[a]) for a in range(3072)], dtype=np.float64)
_vec = np.array(hp.pix2vec(16, np.arange(3072))).T; ANTIPODE = hp.vec2pix(16, -_vec[:, 0], -_vec[:, 1], -_vec[:, 2]); PLANE = np.minimum(np.arange(3072), ANTIPODE)
GATES['G_antipode_map'] = (asha(ANTIPODE.astype(np.int32)) == '11efe112b8f388b851b5218fa1287aafa3e32ce359f98cccb7c6649d79cc599a')
PRIM, SENS = PROD['primary'], PROD['sensitivity']
def packed_into(x, out):                                                                                  # A8b child (verbatim): products in the selection dtype
    k = 0; d = x.shape[1]
    for i in range(d):
        w = d - i; np.multiply(x[:, i:i + 1], x[:, i:], out=out[:, k:k + w]); k += w
    return out
def scan(X, selection):
    """A8b kernel: x64 -> cast to selection dtype -> packed products in that dtype -> GEMM with Fp(selection dtype) -> argmin (first occurrence);
    evaluation: float64 einsum with x64 and the float64 B-stack at the selected axis (A8b eval64). Chunk = A8b winner of that dtype. Returns T1, T2, axis, plane-folded axis."""
    n = len(X); T1 = np.empty(n); T2 = np.empty(n); AX = np.empty(n, np.int32); dt = np.float32 if selection == 'float32' else np.float64; Fp = Fp32 if selection == 'float32' else Fp64; CHUNK = CHUNK_BY_SEL[selection]
    for a in range(0, n, CHUNK):
        x64 = X[a:a + CHUNK]; x = x64 if dt == np.float64 else x64.astype(np.float32); f = packed_into(x, np.empty((len(x), len(iu[0])), dt)); Sp = f @ Fp.T; ax = Sp.argmin(1).astype(np.int32); sel = Sp[np.arange(len(x)), ax]
        t1 = np.einsum('ni,nij,nj->n', x64, Bp[ax], x64, optimize=True); t2 = np.einsum('ni,nij,nj->n', x64, Bm[ax], x64, optimize=True)
        if not (np.isfinite(sel).all() and np.isfinite(t1).all() and np.isfinite(t2).all()): raise FloatingPointError('non-finite selection/evaluation output (A8b child fail-fast semantics)')   # NaN would silently become Event B = False
        AX[a:a + CHUNK] = ax; T1[a:a + CHUNK] = t1; T2[a:a + CHUNK] = t2
    return dict(T1=T1, T2=T2, AX=AX, PL=PLANE[AX])
# regression against the A8b child kernel, re-implemented from the frozen child source (cast / packed_into / select l24_feature231 / eval64) on a 2000-sample probe: bit-identical argmin, T1, T2
def _a8b_child_reference(x64, sel_dtype):
    def cast(x64): return x64 if sel_dtype == np.float64 else x64.astype(np.float32)
    Bp4_64, Bm4_64 = Bp, Bm; Bp4 = Bp4_64 if sel_dtype == np.float64 else Bp4_64.astype(np.float32); Fp = np.array([fvec(Bp4[a]) for a in range(3072)], dtype=sel_dtype)
    x = cast(x64); n = len(x); ar = np.arange(n); f = packed_into(x, np.empty((n, len(iu[0])), sel_dtype)); Sp = f @ Fp.T; a = Sp.argmin(1)
    x4 = x64[:, :21]; return a, np.einsum('ni,nij,nj->n', x4, Bp4_64[a], x4, optimize=True), np.einsum('ni,nij,nj->n', x4, Bm4_64[a], x4, optimize=True)
xt = np.random.default_rng(1).standard_normal((2000, 21)); ok_reg = True
for sel, dt in (('float32', np.float32), ('float64', np.float64)):
    d = scan(xt, sel); a_ref, t1_ref, t2_ref = _a8b_child_reference(xt, dt); ok_reg &= bool(np.array_equal(d['AX'], a_ref) and np.array_equal(d['T1'], t1_ref) and np.array_equal(d['T2'], t2_ref))
GATES['G_a8b_kernel_exact_regression'] = ok_reg
d64 = scan(xt[:5], 'float64')
GATES['G_scan_vs_direct'] = bool(max(abs(d64['T1'][i] - min(xt[i] @ Bp[a] @ xt[i] for a in range(3072))) / abs(d64['T1'][i]) for i in range(5)) < 1e-12 and max(abs(d64['T2'][i] - xt[i] @ Bm[d64['AX'][i]] @ xt[i]) / abs(d64['T2'][i]) for i in range(5)) < 1e-12)
# ---- generator: 1R : m z, CRN across systems ----
MASTER_SEED = 20260912; STREAM = dict(gaussian=0, rotation=1, calibration=2, pseudo=3, bootstrap=4, w2=5)
GEN_NS = dict(calibration=100, m_sensitivity=200, pseudo=300, negative_control=400, w2_independent=500, w2_crn=600, w2_isotropic=700)   # purpose namespaces for generation streams (rotation + gaussian)
GEN_CALLS = []                                                                                                                              # every generate() call: (purpose namespace, ids) -> must be unique
def rng_for(stream, *ids): return np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, STREAM[stream], *[int(i) for i in ids]]))
def generate(N, m, stream_ids, S_list, selections=('float32',), chunk_clusters=2000):
    assert stream_ids[0] in GEN_NS.values(), stream_ids; GEN_CALLS.append(dict(ns=[k for k, v in GEN_NS.items() if v == stream_ids[0]][0], ids=[int(i) for i in stream_ids], N=int(N), m=int(m), systems=len(S_list)))
    K = N // m; assert K * m == N; rr = rng_for('rotation', *stream_ids); rz = rng_for('gaussian', *stream_ids); cid = np.repeat(np.arange(K), m); t_rot = 0.0
    out = {(s, sel): dict(T1=np.empty(N), T2=np.empty(N), AX=np.empty(N, np.int32), PL=np.empty(N, np.int32)) for s in range(len(S_list)) for sel in selections}
    for k0 in range(0, K, chunk_clusters):
        k1 = min(K, k0 + chunk_clusters); t = time.perf_counter(); Rs = Rotation.random(num=k1 - k0, rng=rr).as_matrix(); Ds = D_batch(Rs); t_rot += time.perf_counter() - t; Z = rz.standard_normal((k1 - k0, m, 21)); sl = slice(k0 * m, k1 * m)
        for s, S in enumerate(S_list):
            X = np.einsum('kab,kmb->kma', Ds @ S, Z).reshape(-1, 21)
            for sel in selections:
                d = scan(X, sel)
                for kk in ('T1', 'T2', 'AX', 'PL'): out[(s, sel)][kk][sl] = d[kk]
    return out, cid, dict(K=K, m=m, rotation_seconds=t_rot)
CFG = dict(smoke=dict(N=20_000, N_CAL=20_000, N_PSEUDO=50, B_BOOT=300, B_NULL=10, N_SUB_W2=(1000, 2000), N_W2_POOL=8000, N_FIT=5000),
           official=dict(N=1_000_000, N_CAL=200_000, N_PSEUDO=200, B_BOOT=2000, B_NULL=200, N_SUB_W2=(2000, 5000), N_W2_POOL=200_000, N_FIT=20_000))[A10_MODE]
# ---- calibration sample (isotropic, stream 'calibration') + A5 map-based null cross-check (medians, P(T1<=obs), P(E_B)) ----
t0 = time.time(); calo, cal_cid, cal_info = generate(CFG['N_CAL'], 100, (GEN_NS['calibration'], 0), [S_I], ('float64', 'float32')); cal = calo[(0, PRIM)]; cal64 = calo[(0, 'float64')]; cal32 = calo[(0, 'float32')]
zA5 = np.load(ASSETS['a5_null'][0]); a5t1, a5t2 = zA5['T1_l2_4'], zA5['T2_l2_4']; rb5 = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, 9]))
ci1 = np.quantile([np.median(rb5.choice(a5t1, len(a5t1))) for _ in range(2000)], [0.025, 0.975]); ci2 = np.quantile([np.median(rb5.choice(a5t2, len(a5t2))) for _ in range(2000)], [0.025, 0.975])
def wilson(k, n, z=1.959964):
    p = k / n; c = (p + z * z / (2 * n)) / (1 + z * z / n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (float(c - h), float(c + h))
a5_p1 = int(np.sum(a5t1 <= T1o)); a5_pB = int(np.sum(event_B(a5t1, a5t2))); w1 = wilson(a5_p1, len(a5t1)); wB = wilson(a5_pB, len(a5t1)); e_p1 = float(np.mean(cal['T1'] <= T1o)); e_pB = float(np.mean(event_B(cal['T1'], cal['T2'])))
GATES['G_iso_engine_matches_A5_null'] = bool(ci1[0] <= np.median(cal['T1']) <= ci1[1] and ci2[0] <= np.median(cal['T2']) <= ci2[1] and w1[0] <= e_p1 <= w1[1] and wB[0] <= e_pB <= wB[1])
TOL_NEAR_TIE = 1e-6; TOL_EVENTB_MISMATCH = 1e-5   # PRE-REGISTERED cross-selection sensitivity bounds (A10-specific). Primary = float64 selection; float32 (A8b registered path) = sensitivity.
# Sandbox discovery (v1.2.3 smoke, 8e4 samples): float32/float64 selection flips (~5e-5 per sample) occur (i) to the exact antipode, where pixelised antipodal representatives can have non-identical B- rows so T2 changes
# by up to ~1e-3 relative, and (ii) to a DIFFERENT plane at near ties (two planes' S+ within float32 resolution). Neither can be bounded by a 1e-5 continuous tolerance; what IS invariant is that a flip only occurs
# when the two candidates' selected S+ differ within float32 resolution (T1 rel diff < 1e-6) and that Event B, the primary output, is essentially unaffected (mismatch fraction << P(E_B) ~ 4e-3).
# Policy (v1.2.4+): float64 selection is the PRIMARY l2-4 path (adopted prospectively before the official run); the A8b float32 path is the SENSITIVITY path. Axis reported plane-folded, raw oriented representative kept; T1/T2 evaluated on the primary representative. A8b's TOL_EVAL=1e-6 is not rewritten.
def compare_selection_paths(d_sens, d_prim):
    """(d_sensitivity, d_primary): the primary path is the reference (relative-difference denominators, antipode of the primary axis); labels are dtype-agnostic."""
    d32, d64 = d_sens, d_prim; same = d32['AX'] == d64['AX']; anti = d32['AX'] == ANTIPODE[d64['AX']]; eb32, eb64 = event_B(d32['T1'], d32['T2']), event_B(d64['T1'], d64['T2']); flip = np.flatnonzero(~same)
    tiny = np.finfo(float).tiny; t1r_all = np.abs(d32['T1'] - d64['T1']) / np.maximum(np.abs(d64['T1']), tiny); t2r_all = np.abs(d32['T2'] - d64['T2']) / np.maximum(np.abs(d64['T2']), tiny); mism = int(np.sum(eb32 != eb64))
    ang = np.degrees(np.arccos(np.clip(np.abs(np.einsum('ij,ij->i', _vec[d32['AX'][flip]], _vec[d64['AX'][flip]])), 0, 1))) if len(flip) else np.zeros(0)   # plane-folded angular distance (deg) between the two selected axes
    return dict(n=int(len(same)), flip_frac=float((~same).mean()), n_flip=int(len(flip)), n_antipodal=int(np.sum(~same & anti)), n_nonantipodal=int(np.sum(~same & ~anti)), T1_rel_max=float(t1r_all.max()), T2_rel_max=float(t2r_all.max()),
                T1_rel_max_at_flips=float(t1r_all[flip].max()) if len(flip) else 0.0, max_plane_angle_deg_nonantipodal=float(ang[~anti[flip]].max()) if np.any(~anti[flip]) else 0.0, flips_only_at_near_ties=bool(len(flip) == 0 or t1r_all[flip].max() < TOL_NEAR_TIE), eventB_mismatch=mism, eventB_mismatch_frac=mism / len(same), eventB_identical=bool(mism == 0),
                all_finite=bool(np.isfinite(d32['T1']).all() and np.isfinite(d32['T2']).all() and np.isfinite(d64['T1']).all() and np.isfinite(d64['T2']).all()),
                consistency=bool(np.isfinite(d32['T1']).all() and np.isfinite(d32['T2']).all() and np.isfinite(d64['T1']).all() and np.isfinite(d64['T2']).all() and (len(flip) == 0 or t1r_all[flip].max() < TOL_NEAR_TIE) and mism / len(same) <= TOL_EVENTB_MISMATCH)), \
           dict(idx=flip.astype(np.int64), AX_sensitivity=d32['AX'][flip], AX_primary=d64['AX'][flip], antipode=anti[flip], plane_angle_deg=ang, T1_sensitivity=d32['T1'][flip], T1_primary=d64['T1'][flip], T2_sensitivity=d32['T2'][flip], T2_primary=d64['T2'][flip], EB_sensitivity=eb32[flip], EB_primary=eb64[flip])
# self-test of the cross-selection helper (5 synthetic cases against the SAME function used by the gates)
def _mk(ax, t1, t2): return dict(AX=np.array(ax, np.int32), T1=np.array(t1, float), T2=np.array(t2, float))
_n = 200000; _ax = np.arange(_n) % 3072; _t1 = np.full(_n, 100.0); _t2 = np.full(_n, 300.0); ref = _mk(_ax, _t1, _t2)
_c1 = compare_selection_paths(_mk(_ax, _t1, _t2), ref)[0]['consistency']                                                              # 1 same axis + same outputs -> PASS   (argument order = (alternative, reference), as in every real call)
_a2 = _ax.copy(); _a2[0] = (_ax[0] + 7) % 3072; _t1b = _t1.copy(); _t1b[0] *= 1 + 1e-8; _c2 = compare_selection_paths(_mk(_a2, _t1b, _t2), ref)[0]['consistency']   # 2 non-antipodal near-tie flip, EB same -> PASS
_t1c = _t1.copy(); _t1c[0] *= 1 + 5e-6; _c3 = compare_selection_paths(_mk(_a2, _t1c, _t2), ref)[0]['consistency']                    # 3 flip with T1 diff >= 1e-6 -> FAIL
_t2d = _t2.copy(); _t2d[:3] = 10.0; _c4 = compare_selection_paths(_mk(_ax, np.full(_n, T1o - 1), np.where(np.arange(_n) < 3, T2o - 1, T2o + 1)), _mk(_ax, np.full(_n, T1o - 1), np.full(_n, T2o + 1)))[0]['consistency']   # 4 EB mismatch rate 3/2e5 > 1e-5 -> FAIL
_t1e = _t1.copy(); _t1e[0] = np.nan; _c5 = compare_selection_paths(_mk(_ax, _t1e, _t2), ref)[0]['consistency']                       # 5 non-finite output -> FAIL
_bad = _mk(_a2, _t1c, _t2); _c6 = compare_selection_paths(_bad, _bad)[0]                                                            # 6 self-comparison of a bad alternative reports NO flip: a self-comparison cannot detect anything -> the alias asserts below guard the real calls
_c6ok = (_c6['n_flip'] == 0 and _c6['consistency'])
GATES['G_cross_selection_helper_selftest'] = bool(_c1 and _c2 and not _c3 and not _c4 and not _c5 and _c6ok); DIAG['cross_selection_selftest'] = dict(same_pass=_c1, near_tie_flip_pass=_c2, large_T1_diff_fail=not _c3, eb_mismatch_rate_fail=not _c4, nonfinite_fail=not _c5, self_comparison_blind=_c6ok)
assert cal is cal64 and cal32 is not cal64 and cal32['AX'] is not cal64['AX'], 'alias check: primary must be float64 and the sensitivity array must be a distinct float32 result'
CALX, cal_flips = compare_selection_paths(cal32, cal64)   # (sensitivity float32, primary float64)
GATES['G_cal_f32_f64_selection_consistency'] = CALX['consistency']
REC['calibration'] = dict(N=CFG['N_CAL'], m=100, primary_selection=PRIM, T1_med=float(np.median(cal['T1'])), T2_med=float(np.median(cal['T2'])), P_T1_le_obs=e_p1, P_eventB=e_pB, T2_q16_q84_secondary=[float(np.quantile(cal['T2'], .16)), float(np.quantile(cal['T2'], .84))], f32_f64_axis_flip_frac=float(np.mean(cal32['AX'] != cal64['AX'])), cross_selection=CALX,
                          A5=dict(n=len(a5t1), T1_med=float(np.median(a5t1)), T1_med_CI=ci1.tolist(), T2_med=float(np.median(a5t2)), T2_med_CI=ci2.tolist(), P_T1_le_obs=a5_p1 / len(a5t1), P_T1_le_obs_wilson=w1, P_eventB=a5_pB / len(a5t1), P_eventB_wilson=wB), seconds=time.time() - t0)
mu_c = np.array([cal['T1'].mean(), cal['T2'].mean()]); Sig_c = np.cov(np.vstack([cal['T1'], cal['T2']])); Sih = np.linalg.inv(np.linalg.cholesky(Sig_c))
GATES['G_cal_sensitivity_wiring'] = bool(CALX['n'] == len(cal32['AX']) and (CALX['n_flip'] == int(np.sum(cal32['AX'] != cal64['AX']))))   # the gate must have compared the float32 result against the float64 primary
atomic_npz(os.path.join(CKPT, 'a10a_calibration.npz'), T1=cal['T1'], T2=cal['T2'], AX=cal['AX'], PL=cal['PL'], cid=cal_cid, T1_f32sel=cal32['T1'], T2_f32sel=cal32['T2'], AX_f32sel=cal32['AX'], **{f'flip_{k}': v for k, v in cal_flips.items()})
REQ_A = ['G_repo_commit', 'G_repo_origin', 'G_repo_clean', 'G_live_modules', 'G_env_versions', 'G_threads_live_registered', 'G_notebook_live_source', 'G_asset_file_sha', 'G_a9_binding', 'G_a9_basis_hashes', 'G_a9_quadrature_hashes', 'G_a8b_production_spec_bound', 'G_a8b_chunk_binding', 'G_a8b_kernel_exact_regression', 'G_a5_binding', 'G_bstack_array_sha', 'G_cvec_sha', 'G_cvec_lblock_constant', 'G_basis_order', 'G_step0_obs_bound', 'G_cov_basis_matches_M21', 'G_cov_positions_distinct', 'G_matched_power', 'G_sqrt_hard',
         'G_quadrature_orthonormal', 'G_D_batch_matches_single', 'G_D_orthogonal', 'G_D_homomorphism', 'G_D_direct_geometry', 'G_D_known_z_rotation', 'G_antipode_map', 'G_scan_vs_direct', 'G_cross_selection_helper_selftest', 'G_iso_engine_matches_A5_null', 'G_cal_f32_f64_selection_consistency', 'G_cal_sensitivity_wiring']
STATUS['ENGINE_VALID'] = all(GATES[k] for k in REQ_A); assert STATUS['ENGINE_VALID'], {k: GATES[k] for k in REQ_A if not GATES[k]}
print(f"A10a ENGINE_VALID | mode {A10_MODE} | MT {MT_COMMIT[:12]} | threads {THREADS_LIVE} | versions {VERS} mismatch {VERS_MISMATCH}\n   calibration: T1_med {REC['calibration']['T1_med']:.1f} (A5 {REC['calibration']['A5']['T1_med']:.1f} CI {np.round(ci1,1).tolist()}) P(T1<=obs) {e_p1:.4f} (A5 {a5_p1}/1000) P(E_B) {e_pB:.4f} (A5 {a5_pB}/1000) f32/f64 flip {REC['calibration']['f32_f64_axis_flip_frac']:.4f} ({REC['calibration']['seconds']:.0f}s)")


In [ ]:
# ---- A10b: orientation-cluster m sensitivity (Event B, production path primary, float64 selection sensitivity, 4 runs, fail-closed decision) ----
M_LIST = (10, 100); REPS = (1, 2); MS = {}; KEEP = {}
def cluster_boot_logQ(hM, hI, K, B, seed):
    rb = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, STREAM['bootstrap'], seed])); out = np.empty(B)
    for b in range(B):
        c = np.bincount(rb.integers(0, K, K), minlength=K); sm, si = c @ hM, c @ hI; out[b] = np.log(sm / si) if (sm > 0 and si > 0) else np.nan
    return out
def analyse(dM, dI, cid, K, tag, seed):
    eM, eI = event_B(dM['T1'], dM['T2']), event_B(dI['T1'], dI['T2']); hM, hI = np.bincount(cid, weights=eM, minlength=K), np.bincount(cid, weights=eI, minlength=K); PM, PI = float(eM.mean()), float(eI.mean())
    Q = PM / PI if PI > 0 else np.inf; lq = float(np.log(Q)) if np.isfinite(Q) and Q > 0 else None
    boots = [cluster_boot_logQ(hM, hI, K, CFG['B_BOOT'], seed * 10 + s) for s in range(5)]; cis = [np.nanquantile(b, [0.025, 0.975]) if np.isfinite(b).sum() > 10 else np.array([np.nan, np.nan]) for b in boots]
    widths = np.array([c[1] - c[0] for c in cis]); ci = cis[0]; se = float(np.nanstd(boots[0])); relh = float((np.exp(ci[1]) - np.exp(ci[0])) / 2 / Q) if lq is not None and np.all(np.isfinite(ci)) else np.inf
    return dict(tag=tag, K=K, P_M=PM, P_I=PI, CI_widths_5seeds=widths.tolist(), hits_M=int(eM.sum()), hits_I=int(eI.sum()), Q=float(Q) if np.isfinite(Q) else None, logQ=lq, se_logQ=se, logCI95=ci.tolist(), CI95=np.exp(ci).tolist(), CI_rel_halfwidth=relh, CI_width_CV_5seeds=float(widths.std() / widths.mean()) if np.all(np.isfinite(widths)) else None,
                event_positive_clusters_M=int((hM > 0).sum()), event_positive_clusters_I=int((hI > 0).sum()), precision_gate=bool((hM > 0).sum() >= 50 and (hI > 0).sum() >= 50 and relh <= 0.20 and np.all(np.isfinite(widths)) and widths.std() / widths.mean() < 0.2),
                P_M_T1_le_obs=float(np.mean(dM['T1'] <= T1o)), P_I_T1_le_obs=float(np.mean(dI['T1'] <= T1o)), T1_med_M=float(np.median(dM['T1'])), T1_med_I=float(np.median(dI['T1'])), T2_med_M=float(np.median(dM['T2'])), T2_med_I=float(np.median(dI['T2']))), hM, hI, boots
BOOTS = {}; BOOTS5 = {}; XSEL = {}; FLIPS = {}; AXES = {}
for m in M_LIST:
    for rep in REPS:
        t = time.time(); out, cid, gi = generate(CFG['N'], m, (GEN_NS['m_sensitivity'], m, rep), [S_M, S_I], ('float64', 'float32')); K = gi['K']
        r, hM, hI, bt = analyse(out[(0, PRIM)], out[(1, PRIM)], cid, K, f'm{m}_rep{rep}', 100 * m + rep); r32, hM32, hI32, _ = analyse(out[(0, SENS)], out[(1, SENS)], cid, K, f'm{m}_rep{rep}_f32sel', 100 * m + rep)   # sensitivity path (A8b float32 selection)
        r.update(rotation_seconds=gi['rotation_seconds'], seconds=time.time() - t, sensitivity_float32_selection=dict(Q=r32['Q'], logQ=r32['logQ'], CI95=r32['CI95'], hits_M=r32['hits_M'], hits_I=r32['hits_I'], abs_dlogQ_primary_vs_f32=(abs(r['logQ'] - r32['logQ']) if (r['logQ'] is not None and r32['logQ'] is not None) else None)))
        for s_, nm in ((0, 'model'), (1, 'iso')): XSEL[f"{r['tag']}_{nm}"], FLIPS[f"{r['tag']}_{nm}"] = compare_selection_paths(out[(s_, SENS)], out[(s_, PRIM)])   # (sensitivity, primary)
        AXES[f"{r['tag']}_model_AX"] = out[(0, PRIM)]['AX'].astype(np.int16); AXES[f"{r['tag']}_iso_AX"] = out[(1, PRIM)]['AX'].astype(np.int16); AXES[f"{r['tag']}_model_AX_{SENS}"] = out[(0, SENS)]['AX'].astype(np.int16); AXES[f"{r['tag']}_iso_AX_{SENS}"] = out[(1, SENS)]['AX'].astype(np.int16)
        r['cross_selection'] = {nm: XSEL[f"{r['tag']}_{nm}"] for nm in ('model', 'iso')}
        MS[r['tag']] = r; BOOTS[r['tag']] = bt[0]; BOOTS5[r['tag']] = bt; KEEP[(m, rep)] = dict(hM=hM, hI=hI, hM32=hM32, hI32=hI32, K=K)
        if rep == 1: KEEP[m] = dict(dM=out[(0, PRIM)], dI=out[(1, PRIM)], dM32=out[(0, SENS)], dI32=out[(1, SENS)], cid=cid, K=K)
        print(f"{r['tag']}: K={K} P_M={r['P_M']:.5f} P_I={r['P_I']:.5f} Q={r['Q']} CI={np.round(r['CI95'],3).tolist()} relh={r['CI_rel_halfwidth']:.3f} ev+ M/I={r['event_positive_clusters_M']}/{r['event_positive_clusters_I']} prec={r['precision_gate']} | f32sel Q={r32['Q']} | {time.time()-t:.0f}s")
atomic_npz(os.path.join(CKPT, 'a10b_cluster_hits.npz'), **{f'm{m}_rep{rep}_{k}': KEEP[(m, rep)][k] for m in M_LIST for rep in REPS for k in ('hM', 'hI', 'hM32', 'hI32')}, **{f'{t}_boot_logQ_seed{i}': b for t, bl in BOOTS5.items() for i, b in enumerate(bl)})
# decision (fail-closed): per-m usable status; difference CIs (guarded by finiteness) from INDEPENDENT bootstrap draws of logQ
prec = {t: MS[t]['precision_gate'] for t in MS}
def m_status(m):
    fin = all(MS[f'm{m}_rep{r}']['logQ'] is not None for r in REPS); ok = all(prec[f'm{m}_rep{r}'] for r in REPS)
    cons = bool(fin and abs(MS[f'm{m}_rep2']['logQ'] - MS[f'm{m}_rep1']['logQ']) <= 1.96 * np.sqrt(MS[f'm{m}_rep1']['se_logQ'] ** 2 + MS[f'm{m}_rep2']['se_logQ'] ** 2)); return dict(finite=fin, precise=ok, consistent=cons, usable=bool(fin and ok and cons))
MST = {m: m_status(m) for m in M_LIST}; m10_ok, m100_ok = MST[10]['usable'], MST[100]['usable']; allQ = MST[10]['finite'] and MST[100]['finite']
def diff_ci(a, b): d = BOOTS[a] - BOOTS[b]; return [float(np.nanquantile(d, 0.025)), float(np.nanquantile(d, 0.975))]
d1, d2 = (diff_ci('m100_rep1', 'm10_rep1'), diff_ci('m100_rep2', 'm10_rep2')) if allQ else ([np.nan, np.nan], [np.nan, np.nan]); dp = 0.5 * ((BOOTS['m100_rep1'] + BOOTS['m100_rep2']) - (BOOTS['m10_rep1'] + BOOTS['m10_rep2'])); dpci = [float(np.nanquantile(dp, 0.025)), float(np.nanquantile(dp, 0.975))] if allQ else [np.nan, np.nan]
DELTA_M = float(np.log(1.10))                                                                                # PRE-REGISTERED practical-equivalence margin: |logQ(100) - logQ(10)| within +-log(1.10)
within = lambda c: bool(np.all(np.isfinite(c)) and c[0] >= -DELTA_M and c[1] <= DELTA_M); equiv = within(d1) and within(d2) and within(dpci)
def _eq_seed(i):
    if not allQ: return None
    q = lambda d: [float(np.nanquantile(d, 0.025)), float(np.nanquantile(d, 0.975))]; a1 = q(BOOTS5['m100_rep1'][i] - BOOTS5['m10_rep1'][i]); a2 = q(BOOTS5['m100_rep2'][i] - BOOTS5['m10_rep2'][i]); ap = q(0.5 * ((BOOTS5['m100_rep1'][i] + BOOTS5['m100_rep2'][i]) - (BOOTS5['m10_rep1'][i] + BOOTS5['m10_rep2'][i])))
    return within(a1) and within(a2) and within(ap)
EQ5 = [_eq_seed(i) for i in range(5)]   # diagnostic: equivalence verdict per bootstrap seed (the decision uses seed 0, pre-registered)
adopted = 100 if (m10_ok and m100_ok and equiv) else (10 if m10_ok else None)                                # policy-aware: m=10 fallback needs only m=10 usable
reason = ('m=100 practically equivalent to m=10 within the pre-registered margin; both usable' if adopted == 100 else ('m=10 adopted: m=100 not usable (finite/precise/consistent) or equivalence not established' if adopted == 10 else 'unresolved: m=10 not usable'))
M_DECISION = dict(rule='m=100 iff m10 and m100 are both usable (finite Q, precision gate on both reps, two-rep consistency) AND the m100-m10 logQ difference CI lies within [-DELTA_M, +DELTA_M] for rep1 pair, rep2 pair and the pooled difference; '
                       'm=10 iff m10 is usable and (m100 not usable OR equivalence not established); otherwise unresolved (fail-closed). A CI containing 0 alone is NOT treated as equivalence.',
                  DELTA_M=DELTA_M, DELTA_M_rationale='Engineering tolerance, not derived: coarsening the orientation clustering to m=100 is accepted if the multiplicative discrepancy of the support ratio Q is within 10%, i.e. below the Phase A Monte Carlo design differences. '
                                                       'Near the decision thresholds Q=3/10 a 10% difference can change a verdict; the full grid (Phase B/C) must additionally record that no threshold crossing arises from the m choice.',
                  per_m=MST, diff_logCI_rep1=d1, diff_logCI_rep2=d2, diff_logCI_pooled=dpci, equivalence_established=bool(equiv), equivalence_verdict_5_bootstrap_seeds=EQ5, zero_in_all_CIs=bool(allQ and all(c[0] <= 0 <= c[1] for c in (d1, d2, dpci))),
                  point_diff_rep1=MS['m100_rep1']['logQ'] - MS['m10_rep1']['logQ'] if allQ else None, point_diff_rep2=MS['m100_rep2']['logQ'] - MS['m10_rep2']['logQ'] if allQ else None,
                  replicate_delta_logQ_m10=abs(MS['m10_rep2']['logQ'] - MS['m10_rep1']['logQ']) if MST[10]['finite'] else None, replicate_delta_logQ_m100=abs(MS['m100_rep2']['logQ'] - MS['m100_rep1']['logQ']) if MST[100]['finite'] else None,
                  precision=prec, adopted_m=adopted, reason=reason, cluster_ESS='superseded: precision gate = event-positive clusters >= 50 (both systems) AND CI relative half-width <= 0.20 AND 5-seed CI-width CV < 0.2')
GATES['G_m_run_inventory'] = (set(MS) == {f'm{m}_rep{r}' for m in M_LIST for r in REPS})
GATES['G_m10_usable'] = m10_ok; GATES['G_m100_usable'] = m100_ok                                            # m100 diagnostics are always saved; m100 usability is required only when m=100 is adopted (inside G_m_decision_policy)
GATES['G_m_decision_policy'] = bool((adopted == 100 and m10_ok and m100_ok and equiv) or (adopted == 10 and m10_ok and (not m100_ok or not equiv)))
GATES['G_m_decision_resolved'] = (adopted is not None)
GATES['G_m_f32_f64_selection_consistency_all_runs'] = all(v['consistency'] for v in XSEL.values()); GATES['G_m_f32_f64_no_nonantipodal_flip'] = all(v['n_nonantipodal'] == 0 for v in XSEL.values())   # second one is DIAGNOSTIC (non-antipodal near-tie flips observed in sandbox)
DLOGQ_F32 = {t: MS[t]['sensitivity_float32_selection']['abs_dlogQ_primary_vs_f32'] for t in MS}; DELTA_Q_SENS = 0.01   # PRE-REGISTERED diagnostic bound on the direct Q sensitivity to the A8b float32 path
GATES['G_m_Q_f32_sensitivity_bound'] = all(v is not None and v <= DELTA_Q_SENS for v in DLOGQ_F32.values())   # DIAGNOSTIC (float32 is not the primary path)
atomic_npz(os.path.join(CKPT, 'a10b_flip_evidence.npz'), **{f'{k}_{kk}': vv for k, v in FLIPS.items() for kk, vv in v.items()}); atomic_npz(os.path.join(CKPT, 'a10b_axes.npz'), **AXES)   # full raw oriented axes of every m run (primary and sensitivity paths; plane-folded label = min(a, ANTIPODE[a]))
REQ_B = ['G_m_run_inventory', 'G_m_f32_f64_selection_consistency_all_runs'] + (['G_m10_usable', 'G_m_decision_policy', 'G_m_decision_resolved'] if A10_MODE == 'official' else []); STATUS['M_SENSITIVITY_RESOLVED'] = all(GATES[k] for k in REQ_B)   # smoke N is too small for the precision gate: recorded, not required
atomic_json(dict(runs=MS, decision=M_DECISION, gates={k: GATES[k] for k in REQ_B}), os.path.join(CKPT, 'a10b_m_sensitivity.json')); print('m decision:', {k: v for k, v in M_DECISION.items() if k not in ('rule', 'cluster_ESS')}, '| status', STATUS['M_SENSITIVITY_RESOLVED'])
if A10_MODE == 'official': assert STATUS['M_SENSITIVITY_RESOLVED'] and M_DECISION['adopted_m'] is not None, M_DECISION   # fail-fast before the expensive components


In [ ]:
# ---- A10c: ONE-POINT calibration PATHWAY prototype (Event B, 2D pseudo thresholds, no re-scan) with negative / positive controls and brute-force equality gate ----
m_use = M_DECISION['adopted_m'] if A10_MODE == 'official' else (M_DECISION['adopted_m'] or 100); assert m_use is not None; dM, dI, cid, K = KEEP[m_use]['dM'], KEEP[m_use]['dI'], KEEP[m_use]['cid'], KEEP[m_use]['K']
(dP,), _, _ = (lambda o, c, g: ([o[(0, PRIM)]], c, g))(*generate(CFG['N_PSEUDO'], 1, (GEN_NS['pseudo'], 0), [S_I], (PRIM,))); pT1, pT2 = dP['T1'], dP['T2']
def hit_table_2d(T1, T2, cid, thr1, thr2):
    """H[k, p] = #{i in cluster k : T1_i <= thr1[p] and T2_i <= thr2[p]} via sort on T1 + cumulative per-cluster counts over the T2 condition (O(P * N) worst case, vectorised per pseudo)."""
    o = np.argsort(T1); T1s, T2s, cs = T1[o], T2[o], cid[o]; H = np.zeros((K, len(thr1)))
    for p in range(len(thr1)):
        n = np.searchsorted(T1s, thr1[p], side='right'); sel = T2s[:n] <= thr2[p]; H[:, p] = np.bincount(cs[:n][sel], minlength=K)
    return H
def brute_table(T1, T2, cid, thr1, thr2):
    H = np.zeros((K, len(thr1)))
    for p in range(len(thr1)): H[:, p] = np.bincount(cid[(T1 <= thr1[p]) & (T2 <= thr2[p])], minlength=K)
    return H
t = time.time(); HM, HI = hit_table_2d(dM['T1'], dM['T2'], cid, pT1, pT2), hit_table_2d(dI['T1'], dI['T2'], cid, pT1, pT2); t_tab = time.time() - t
nb = min(10, len(pT1)); GATES['G_cal_table_matches_bruteforce'] = bool(np.array_equal(HM[:, :nb], brute_table(dM['T1'], dM['T2'], cid, pT1[:nb], pT2[:nb])) and np.array_equal(HI[:, :nb], brute_table(dI['T1'], dI['T2'], cid, pT1[:nb], pT2[:nb])))
rb = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 11, STREAM['bootstrap'], 777])); Wc = np.zeros((CFG['B_BOOT'], K)); Wc2 = np.zeros((CFG['B_BOOT'], K))
for b in range(CFG['B_BOOT']): Wc[b] = np.bincount(rb.integers(0, K, K), minlength=K); Wc2[b] = np.bincount(rb.integers(0, K, K), minlength=K)
def q_lower_ci(HM_, HI_, paired=True):
    # paired cluster bootstrap (same weights for numerator and denominator) or independent (negative control). Zero numerator is kept as Q=0 (lower-tail evidence);
    # zero denominator with positive numerator -> +inf; both zero -> undefined (excluded, counted). Lower CI = 2.5% quantile over the defined replicates.
    SM, SI = Wc @ HM_, (Wc if paired else Wc2) @ HI_; Qb = np.full(SM.shape, np.nan); den = SI > 0; Qb[den] = SM[den] / SI[den]; Qb[(SI == 0) & (SM > 0)] = np.inf
    valid = np.isfinite(Qb) | np.isinf(Qb); frac = dict(valid=valid.mean(0), den_zero=(SI == 0).mean(0), both_zero=((SI == 0) & (SM == 0)).mean(0), num_zero=(SM == 0).mean(0))
    lo = np.array([np.quantile(Qb[valid[:, p], p], 0.025) if valid[:, p].sum() >= 0.95 * len(Qb) else np.nan for p in range(HM_.shape[1])]); return lo, (HM_.sum(0) / len(dM['T1'])) / np.maximum(HI_.sum(0) / len(dI['T1']), 1e-300), frac
lo, Qp, FR = q_lower_ci(HM, HI)
from scipy.stats import gaussian_kde
rf = rng_for('calibration', 5); fM = rf.choice(len(dM['T1']), CFG['N_FIT'], replace=False); fI = rf.choice(len(dI['T1']), CFG['N_FIT'], replace=False)
kM = gaussian_kde(np.vstack([dM['T1'][fM], dM['T2'][fM]])); kI = gaussian_kde(np.vstack([dI['T1'][fI], dI['T2'][fI]])); Dp = kM(np.vstack([pT1, pT2])) / np.maximum(kI(np.vstack([pT1, pT2])), 1e-300)
support = (lo >= 3) & (np.log(Dp) > 0); n_p = len(pT1); k_s = int(np.nansum(support))
# sensitivity of the pathway to the A8b float32 selection path: same clusters, same pseudo thresholds, hit tables from the float32-selected outputs
dM32, dI32 = KEEP[m_use]['dM32'], KEEP[m_use]['dI32']; HM32, HI32 = hit_table_2d(dM32['T1'], dM32['T2'], cid, pT1, pT2), hit_table_2d(dI32['T1'], dI32['T2'], cid, pT1, pT2); lo32, Qp32, FR32 = q_lower_ci(HM32, HI32)
kM32 = gaussian_kde(np.vstack([dM32['T1'][fM], dM32['T2'][fM]])); kI32 = gaussian_kde(np.vstack([dI32['T1'][fI], dI32['T2'][fI]])); Dp32 = kM32(np.vstack([pT1, pT2])) / np.maximum(kI32(np.vstack([pT1, pT2])), 1e-300); support32 = (lo32 >= 3) & (np.log(Dp32) > 0)
CAL_SENS32 = dict(support_mismatch_count=int(np.sum(support != support32)), false_support_frequency_f32=float(np.nanmean(support32)), hit_table_cells_changed_M=int(np.sum(HM != HM32)), hit_table_cells_changed_I=int(np.sum(HI != HI32)), max_abs_dlogQ_pseudo=float(np.nanmax(np.abs(np.log(np.maximum(Qp, 1e-300)) - np.log(np.maximum(Qp32, 1e-300))))), max_abs_dlogD_pseudo=float(np.nanmax(np.abs(np.log(Dp) - np.log(Dp32)))))
# controls: negative = isotropic 'model' (S_I vs S_I clusters -> Q==1 exactly under CRN... use an independent iso replicate instead); positive = synthetic boosted model (T1 scaled by 0.5 -> Event B probability strongly increased)
(dN,), cidN, giN = (lambda o, c, g: ([o[(0, PRIM)]], c, g))(*generate(CFG['N'], m_use, (GEN_NS['negative_control'], m_use), [S_I], (PRIM,))); HN = hit_table_2d(dN['T1'], dN['T2'], cidN, pT1, pT2); loN, QN, FRN = q_lower_ci(HN, HI, paired=False)
kN = gaussian_kde(np.vstack([dN['T1'][fI], dN['T2'][fI]])); DN = kN(np.vstack([pT1, pT2])) / np.maximum(kI(np.vstack([pT1, pT2])), 1e-300); neg_support = (loN >= 3) & (np.log(DN) > 0)
HB = hit_table_2d(0.35 * dI['T1'], 0.35 * dI['T2'], cid, pT1, pT2); loB, QB, FRB = q_lower_ci(HB, HI); kB = gaussian_kde(np.vstack([0.35 * dI['T1'][fI], 0.35 * dI['T2'][fI]])); DB = kB(np.vstack([pT1, pT2])) / np.maximum(kI(np.vstack([pT1, pT2])), 1e-300); pos_support = (loB >= 3) & (np.log(DB) > 0)
HIo = hit_table_2d(dI['T1'], dI['T2'], cid, np.array([T1o]), np.array([T2o])); HBo = hit_table_2d(0.35 * dI['T1'], 0.35 * dI['T2'], cid, np.array([T1o]), np.array([T2o])); loBo, QBo, FRBo = q_lower_ci(HBo, HIo); DBo = float(kB(np.array([[T1o], [T2o]]))[0] / max(kI(np.array([[T1o], [T2o]]))[0], 1e-300))
# negative control: an independent isotropic replicate must not trigger support at the pseudo thresholds; positive control (T1, T2 scaled by 0.35): at the OBSERVED thresholds (a tail event, P_iso ~ 4e-3) the boosted sample must trigger support with margin
GATES['G_cal_negative_control'] = bool(np.nanmean(neg_support) <= 0.05); GATES['G_cal_positive_control'] = bool(np.isfinite(QBo[0]) and np.isfinite(loBo[0]) and np.isfinite(DBo) and HIo.sum() > 0 and HBo.sum() > 0 and loBo[0] >= 3 and QBo[0] >= 10 and DBo > 1)
GATES['G_cal_main_inventory'] = bool(len(pT1) == CFG['N_PSEUDO'] == len(Qp) == len(lo) == len(Dp) == len(support) and np.all(np.isfinite(Qp)) and np.all(np.isfinite(lo)) and np.all(np.isfinite(Dp)) and np.all(Dp > 0))
GATES['G_cal_bootstrap_valid_fraction'] = bool(np.all(FR['valid'] >= 0.95) and np.all(FRN['valid'] >= 0.95) and np.all(FRB['valid'] >= 0.95) and FRBo['valid'][0] >= 0.95)
GATES['G_cal_control_inventory'] = bool(len(QN) == len(loN) == len(DN) == CFG['N_PSEUDO'] and np.all(np.isfinite(QN)) and np.all(np.isfinite(loN)) and np.all(np.isfinite(DN)) and np.all(DN > 0)
                                        and len(QB) == len(loB) == len(DB) == CFG['N_PSEUDO'] and np.all(np.isfinite(QB)) and np.all(np.isfinite(loB)) and np.all(np.isfinite(DB)) and np.all(DB > 0))
CAL = dict(status_name='one_point_one_system_calibration_pathway_prototype', NOT='familywise global calibration (requires all families, both systems, family prior integration, n_pseudo >= 2000 or precision stopping; Phase B/C engine)',
           model_point='E7_b1_A x0(1), PR3-power-matched', m=m_use, n_pseudo=n_p, event='Event B with 2D pseudo thresholds (T1 <= T1_pseudo and T2 <= T2_pseudo)', method='sorted-T1 searchsorted + per-cluster counts; one shared cluster-resampling matrix (B x K) for every pseudo; Q lower CI = 2.5% percentile; D = KDE point estimate (fitting subsample separate from evaluation)',
           one_point_false_support_frequency=k_s / n_p, one_point_false_support_wilson=wilson(k_s, n_p), one_point_strong_raw_frequency=float(np.nanmean((lo >= 10) & (Dp > 1))),
           negative_control=dict(support_rate=float(np.nanmean(neg_support)), Q_median=float(np.nanmedian(QN)), bootstrap='independent numerator/denominator weights'), bootstrap_fractions=dict(valid_min=float(FR['valid'].min()), den_zero_max=float(FR['den_zero'].max()), num_zero_max=float(FR['num_zero'].max()), both_zero_max=float(FR['both_zero'].max())),
           positive_control=dict(support_rate_at_pseudo=float(np.nanmean(pos_support)), Q_median_at_pseudo=float(np.nanmedian(QB)), Q_at_observed=float(QBo[0]), Q_lowerCI_at_observed=float(loBo[0]), D_at_observed=DBo, note='at central pseudo thresholds the ratio is capped by 1/P_iso, so the control is evaluated at the observed (tail) thresholds', construction='isotropic sample with T1 and T2 scaled by 0.35 (synthetic boosted Event B), same clusters'),
           sensitivity_float32_selection=dict(scope='CONDITIONAL: primary (float64) pseudo thresholds held fixed; model/isotropic hit tables, Q lower CI, D and support recomputed from the float32-selected outputs (same clusters)', **CAL_SENS32), Q_pseudo_median=float(np.nanmedian(Qp)), pseudo_T1_q=np.quantile(pT1, [0.05, 0.5, 0.95]).tolist(), pseudo_T2_q=np.quantile(pT2, [0.05, 0.5, 0.95]).tolist(), table_seconds=t_tab)
REQ_C = ['G_cal_table_matches_bruteforce', 'G_cal_negative_control', 'G_cal_positive_control', 'G_cal_main_inventory', 'G_cal_control_inventory', 'G_cal_bootstrap_valid_fraction']; STATUS['CALIBRATION_PATH_VALID'] = all(GATES[k] for k in REQ_C)
atomic_npz(os.path.join(CKPT, 'a10c_pathway.npz'), pseudo_T1=pT1, pseudo_T2=pT2, Q=Qp, Q_lowerCI=lo, D=Dp, Q_neg=QN, Q_neg_lowerCI=loN, Q_pos=QB, Q_pos_lowerCI=loB); atomic_json(dict(CAL=CAL, gates={k: GATES[k] for k in REQ_C}), os.path.join(CKPT, 'a10c_pathway.json'))
print('A10c pathway:', {k: CAL[k] for k in ('one_point_false_support_frequency', 'negative_control', 'positive_control', 'Q_pseudo_median')}, '| status', STATUS['CALIBRATION_PATH_VALID'])


In [ ]:
# ---- A10d: W2 pathway on real engine output (float64 primary selection). PRIMARY: three observer positions from INDEPENDENT (R,z) streams, null = three independent isotropic samples (same sampling mechanism -> finite-pool exceedance estimate).
#           DIAGNOSTIC: CRN three-position W2 (same (R,z) clusters for all positions) against the same null = conservative, variance-reduced reference (NOT an exact MC p). Pre-registered stability gates. ----
def white(d): return (np.column_stack([d['T1'], d['T2']]) - mu_c) @ Sih.T
def w2_exact(a, b):
    Mc = ot.dist(a, b, metric='sqeuclidean'); v, lg = ot.emd2(np.full(len(a), 1 / len(a)), np.full(len(b), 1 / len(b)), Mc, numItermax=1_000_000, log=True); out = float(np.sqrt(v))
    if not np.isfinite(out): raise FloatingPointError('non-finite W2')
    return out, int(bool(lg.get('warning')))
m_w2 = m_use; STAB_SPREAD = 0.25                                                                             # PRE-REGISTERED: max/min - 1 of W2_max across the 3 subsample seeds
t = time.time(); IND = []; IND_CID = []; IND32 = []
for sidx in range(3):                                                                                    # independent streams per position (primary)
    o, c, g = generate(CFG['N_W2_POOL'], m_w2, (GEN_NS['w2_independent'], sidx), [S_POS[sidx]], (PRIM, SENS)); IND.append(white(o[(0, PRIM)])); IND_CID.append(c); IND32.append(white(o[(0, SENS)]))
Kp = g['K']; crn_out, crn_cid, _ = generate(CFG['N_W2_POOL'], m_w2, (GEN_NS['w2_crn'], 0), S_POS, (PRIM, SENS)); CRN = [white(crn_out[(s_, PRIM)]) for s_ in range(3)]; CRN32 = [white(crn_out[(s_, SENS)]) for s_ in range(3)]       # CRN diagnostic (same (R,z) for the three positions)
iso_out, iso_cid, gi2 = generate(3 * CFG['N_W2_POOL'], m_w2, (GEN_NS['w2_isotropic'], 0), [S_I], (PRIM, SENS)); TwI = white(iso_out[(0, PRIM)]); TwI32 = white(iso_out[(0, SENS)]); Ki = gi2['K']; t_gen = time.time() - t
# coupling upper bound on the W2 sensitivity to the float32 selection path: eps_j = sqrt(mean ||z_primary - z_f32||^2) >= W2(P_j, P_j^f32); by the triangle inequality |W2(P_j,P_k) - W2(P_j^f32,P_k^f32)| <= eps_j + eps_k
eps_ind = [float(np.sqrt(np.mean(np.sum((IND[s_] - IND32[s_]) ** 2, axis=1)))) for s_ in range(3)]; eps_crn = [float(np.sqrt(np.mean(np.sum((CRN[s_] - CRN32[s_]) ** 2, axis=1)))) for s_ in range(3)]; eps_iso = float(np.sqrt(np.mean(np.sum((TwI - TwI32) ** 2, axis=1))))
# full-pool RMS is a DIAGNOSTIC only (it does not bound an arbitrary subsample); the required gate uses exact per-subsample bounds and a decision margin (below)
def take(Tw, cid, clusters): return Tw[np.isin(cid, clusters)]
def eps_subset(Tw, Tw32, cid, clusters):
    sel = np.isin(cid, clusters); return float(np.sqrt(np.mean(np.sum((Tw[sel] - Tw32[sel]) ** 2, axis=1))))   # paired-coupling RMS on the EXACT subset fed to OT: >= W2(P_subset^primary, P_subset^f32)
def pairs3(S3):
    pw = {}; w = 0
    for a in range(3):
        for b in range(a + 1, 3): v, ww = w2_exact(S3[a], S3[b]); pw[f'P{a+1}P{b+1}'] = v; w += ww
    return pw, w
W2 = {}; warn = 0; t = time.time()
for n_sub in CFG['N_SUB_W2']:
    ck = os.path.join(CKPT, f'a10d_w2_nsub{n_sub}.json')
    W2_BIND_PAYLOAD = dict(notebook_source=NB_HEAD, live_source=NB_LIVE, versions=VERS, threads=THREADS, threads_live=THREADS_LIVE, platform=platform.platform(), cpu=platform.processor(), asset_sha=ASSET_SHA, IND_sha=[asha(x) for x in IND], CRN_sha=[asha(x) for x in CRN], TwI_sha=asha(TwI), config=CFG, seed=MASTER_SEED, adopted_m=m_w2, production=PROD, mt_commit=MT_COMMIT, mu_c_sha=asha(mu_c), Sih_sha=asha(Sih), S_POS_sha=[asha(S) for S in S_POS], S_I_sha=asha(S_I), mode=A10_MODE, n_sub=n_sub)
    BIND = hashlib.sha256(json.dumps(W2_BIND_PAYLOAD, sort_keys=True, default=_jsonable).encode()).hexdigest(); warn_nsub = 0
    if os.path.exists(ck):
        prev = json.load(open(ck))
        if prev.get('binding') == BIND and prev.get('binding_payload') == json.loads(json.dumps(W2_BIND_PAYLOAD, default=_jsonable)) and prev.get('n_sub') == n_sub: W2[str(n_sub)] = prev; warn += prev['emd_warnings_nsub']; print(f'W2 n_sub={n_sub}: resumed from checkpoint (binding hash + payload matched)'); continue
        print(f'W2 n_sub={n_sub}: stale checkpoint ignored (binding mismatch)')
    kc = n_sub // m_w2; rw = rng_for('w2', n_sub); obs_ind, obs_crn = {}, {}
    for seed in range(3):
        ks = [rw.choice(Kp, kc, replace=False) for s_ in range(3)]; S3 = [take(IND[s_], IND_CID[s_], ks[s_]) for s_ in range(3)]; pw, w = pairs3(S3); warn_nsub += w; e3 = [eps_subset(IND[s_], IND32[s_], IND_CID[s_], ks[s_]) for s_ in range(3)]
        obs_ind[f'seed{seed}'] = dict(pairwise=pw, W2_max=max(pw.values()), eps_subset=e3, W2max_perturbation_bound=max(e3[a] + e3[b] for a in range(3) for b in range(a + 1, 3)))   # |W2max_primary - W2max_f32| <= max_pairs (eps_j + eps_k)
        kk = rw.choice(Kp, kc, replace=False); pw, w = pairs3([take(CRN[s_], crn_cid, kk) for s_ in range(3)]); warn_nsub += w; e3c = [eps_subset(CRN[s_], CRN32[s_], crn_cid, kk) for s_ in range(3)]; obs_crn[f'seed{seed}'] = dict(pairwise=pw, W2_max=max(pw.values()), eps_subset=e3c, W2max_perturbation_bound=max(e3c[a] + e3c[b] for a in range(3) for b in range(a + 1, 3)))
    null = []; null_bound = []; perm = rw.permutation(Ki); ptr = 0
    for b in range(CFG['B_NULL']):                                                                        # each replicate: three disjoint cluster blocks from the isotropic pool
        if ptr + 3 * kc > Ki: perm = rw.permutation(Ki); ptr = 0
        blocks = [perm[ptr + jj * kc: ptr + (jj + 1) * kc] for jj in range(3)]; ptr += 3 * kc; pw, w = pairs3([take(TwI, iso_cid, bl) for bl in blocks]); warn_nsub += w; assert all(np.isfinite(list(pw.values()))); null.append(max(pw.values()))
        eb = [eps_subset(TwI, TwI32, iso_cid, bl) for bl in blocks]; null_bound.append(max(eb[a] + eb[c] for a in range(3) for c in range(a + 1, 3)))
    null = np.array(null); q99 = float(np.quantile(null, 0.99, method='higher')); mcp = lambda v: float((1 + np.sum(null >= v)) / (len(null) + 1)); delta_q99_bound = float(max(null_bound))   # every null W2_max moves by <= its bound, hence the empirical q99 moves by <= max bound
    margin_ok = [bool(abs(obs_ind[f'seed{s_}']['W2_max'] - q99) > obs_ind[f'seed{s_}']['W2max_perturbation_bound'] + delta_q99_bound) for s_ in range(3)]
    warn += warn_nsub; W2[str(n_sub)] = dict(n_sub=n_sub, clusters_per_sample=kc, selection_sensitivity=dict(delta_q99_bound=delta_q99_bound, null_bound_median=float(np.median(null_bound)), decision_margin_ok_per_seed=margin_ok, rule='|W2_max - q99| > observed perturbation bound + delta_q99_bound => the above/below-q99 decision cannot change under the float32 selection path'),
                          primary_independent=dict(observed=obs_ind, finite_pool_exceedance=[mcp(obs_ind[f'seed{s_}']['W2_max']) for s_ in range(3)], decision_above_q99=[bool(obs_ind[f'seed{s_}']['W2_max'] > q99) for s_ in range(3)]),
                          diagnostic_crn=dict(observed=obs_crn, conservative_exceedance=[float(np.mean(null >= obs_crn[f'seed{s_}']['W2_max'])) for s_ in range(3)], note='CRN estimator vs independent-sample null: conservative scale reference, not an exact MC p'),
                          binding=BIND, binding_payload=json.loads(json.dumps(W2_BIND_PAYLOAD, default=_jsonable)), emd_warnings_nsub=warn_nsub, null=dict(B=len(null), q99_method_higher=q99, q95=float(np.quantile(null, 0.95, method='higher')), median=float(np.median(null)), max=float(null.max()), pool_reuse='finite pool; blocks disjoint within a replicate, re-permuted when exhausted'), null_values=null.tolist(), emd_warnings=warn)
    atomic_json(W2[str(n_sub)], ck)
    print(f"W2 n_sub={n_sub}: primary W2_max {[round(obs_ind[f'seed{s_}']['W2_max'],4) for s_ in range(3)]} finite-pool exceedance {W2[str(n_sub)]['primary_independent']['finite_pool_exceedance']} | CRN W2_max {[round(obs_crn[f'seed{s_}']['W2_max'],4) for s_ in range(3)]} | null q99 {q99:.4f} median {np.median(null):.4f} ({time.time()-t:.0f}s)")
def spread(o): v = [o[f'seed{s_}']['W2_max'] for s_ in range(3)]; return max(v) / min(v) - 1
GATES['G_w2_solver_clean'] = (warn == 0 and all(np.all(np.isfinite(W2[k]['null_values'])) for k in W2)); GATES['G_w2_nsub_inventory'] = (set(W2) == {str(n) for n in CFG['N_SUB_W2']})
GATES['G_w2_observed_finite'] = all(np.isfinite([W2[k][p]['observed'][f'seed{s_}']['W2_max'] for k in W2 for p in ('primary_independent', 'diagnostic_crn') for s_ in range(3)]).all() for _ in [0])
GATES['G_w2_pair_seed_inventory'] = all(set(W2[k][p]['observed']) == {'seed0', 'seed1', 'seed2'} and all(set(W2[k][p]['observed'][sd]['pairwise']) == {'P1P2', 'P1P3', 'P2P3'} for sd in W2[k][p]['observed']) for k in W2 for p in ('primary_independent', 'diagnostic_crn'))
GATES['G_w2_null_length_exact'] = all(len(W2[k]['null_values']) == CFG['B_NULL'] for k in W2)
_q99ref = W2[str(max(CFG['N_SUB_W2']))]['null']['q99_method_higher']; W2_SENS = dict(full_pool_rms_diagnostic=dict(eps_independent=eps_ind, eps_crn=eps_crn, eps_isotropic=eps_iso, note='engineering diagnostic on the full empirical pools; does NOT bound an arbitrary subsample'), null_q99_reference=_q99ref,
               exact_subsample=dict(rule='per exact OT subset: eps = paired RMS >= W2(subset_primary, subset_f32); |dW2_max| <= max_pairs(eps_j + eps_k); null q99 shift <= max over replicates of its bound', per_nsub={k: dict(delta_q99_bound=W2[k]['selection_sensitivity']['delta_q99_bound'], observed_bounds=[W2[k]['primary_independent']['observed'][f'seed{s_}']['W2max_perturbation_bound'] for s_ in range(3)], decision_margin_ok=W2[k]['selection_sensitivity']['decision_margin_ok_per_seed']) for k in W2}))
GATES['G_w2_full_pool_selection_rms_small'] = bool(max(eps_ind + eps_crn + [eps_iso]) <= 0.05 * _q99ref)   # DIAGNOSTIC
GATES['G_w2_selection_decision_margin'] = all(all(W2[k]['selection_sensitivity']['decision_margin_ok_per_seed']) for k in W2)   # REQUIRED in official: the above/below-q99 decision is invariant to the float32 selection path (exact subsample bounds)
GATES['G_w2_all_pair_values_finite'] = all(np.isfinite(list(W2[k][p]['observed'][sd]['pairwise'].values())).all() for k in W2 for p in ('primary_independent', 'diagnostic_crn') for sd in W2[k][p]['observed'])
GATES['G_w2_seed_decision_agree'] = all(len(set(W2[k]['primary_independent']['decision_above_q99'])) == 1 for k in W2)
GATES['G_w2_nsub_decision_agree'] = (len({W2[k]['primary_independent']['decision_above_q99'][0] for k in W2}) == 1)
GATES['G_w2_seed_spread'] = all(spread(W2[k]['primary_independent']['observed']) <= STAB_SPREAD for k in W2)
REC['w2'] = dict(estimator='exact 2D W2 (POT emd2, sqeuclidean, sqrt), uniform weights', pot_version=ot.__version__, whitening='calibration sample mean/cov', positions_mock={k: X0[k] for k in X0}, positions_note='mock 3-position set from the A11 cache (E7_b1_A base, E7_b2_A base, y-shift 0.12); NOT the registered p^(3) points',
                primary='independent (R,z) streams per position; null = three disjoint isotropic blocks per replicate (same mechanism); finite-pool Monte Carlo exceedance estimate = (1 + #null >= obs)/(B+1) (null replicates re-use one finite isotropic pool, hence NOT an exact exchangeable MC p); threshold quantile method=higher', diagnostic='CRN three-position W2 (conservative reference)',
                stability_pre_registered=dict(seed_decisions_agree=True, nsub_decisions_agree=True, max_seed_spread=STAB_SPREAD, selection_decision_margin=True), selection_path_sensitivity=dict(**W2_SENS, primary=PRIM, sensitivity=SENS), m=m_w2, B_null=CFG['B_NULL'], results={k: {kk: vv for kk, vv in v.items() if kk != 'null_values'} for k, v in W2.items()}, generation_seconds=t_gen,
                caveat='B=200 99th percentile is coarse (top ~2 values); final trigger threshold requires larger B or precision stopping; this is a pathway result, not a frozen threshold')
REQ_D = ['G_w2_solver_clean', 'G_w2_nsub_inventory', 'G_w2_observed_finite', 'G_w2_all_pair_values_finite', 'G_w2_pair_seed_inventory', 'G_w2_null_length_exact'] + (['G_w2_seed_decision_agree', 'G_w2_nsub_decision_agree', 'G_w2_seed_spread', 'G_w2_selection_decision_margin'] if A10_MODE == 'official' else []); STATUS['W2_PATHWAY_VALID'] = all(GATES[k] for k in REQ_D)   # stability gates: pre-registered, required in official (smoke B/n_sub too small)
atomic_npz(os.path.join(CKPT, 'a10d_w2.npz'), **{f'null_nsub{k}': np.array(v['null_values']) for k, v in W2.items()}); atomic_json(dict(w2=REC['w2'], gates={k: GATES[k] for k in REQ_D}), os.path.join(CKPT, 'a10d_w2.json')); print('A10d status', STATUS['W2_PATHWAY_VALID'])


In [ ]:
# ---- A10e: provenance (component statuses -> A10_VALID), output SHA, atomic write, final assert ----
EXPECTED_OUTPUTS = ['a10a_calibration.npz', 'a10b_cluster_hits.npz', 'a10b_flip_evidence.npz', 'a10b_axes.npz', 'a10b_m_sensitivity.json', 'a10c_pathway.npz', 'a10c_pathway.json', 'a10d_w2.npz', 'a10d_w2.json'] + [f'a10d_w2_nsub{n}.json' for n in CFG['N_SUB_W2']]
outputs = {f: sha(os.path.join(CKPT, f)) for f in EXPECTED_OUTPUTS}; GATES['G_output_inventory_exact'] = (sorted(f for f in os.listdir(CKPT) if not f.endswith('.tmp')) == sorted(EXPECTED_OUTPUTS))
EXPECTED_CALLS = {(100, 0), (200, 10, 1), (200, 10, 2), (200, 100, 1), (200, 100, 2), (300, 0), (400, m_use), (500, 0), (500, 1), (500, 2), (600, 0), (700, 0)}
GATES['G_generation_stream_keys_unique'] = (len({tuple(c['ids']) for c in GEN_CALLS}) == len(GEN_CALLS) and {tuple(c['ids']) for c in GEN_CALLS} == EXPECTED_CALLS)   # unique AND exact 12-call inventory
REQUIRED = REQ_A + REQ_B + REQ_C + REQ_D; INVENTORY = set(REQ_A) | {'G_m_run_inventory', 'G_m_f32_f64_selection_consistency_all_runs', 'G_m_f32_f64_no_nonantipodal_flip', 'G_m_Q_f32_sensitivity_bound', 'G_m10_usable', 'G_m100_usable', 'G_m_decision_policy', 'G_m_decision_resolved'} | set(REQ_C) | set(REQ_D) | {'G_w2_seed_decision_agree', 'G_w2_nsub_decision_agree', 'G_w2_seed_spread', 'G_w2_selection_decision_margin', 'G_w2_full_pool_selection_rms_small', 'G_output_inventory_exact', 'G_generation_stream_keys_unique', 'G_gate_inventory_exact'}
GATES['G_gate_inventory_exact'] = (set(GATES) | {'G_gate_inventory_exact'} == INVENTORY); REQUIRED = REQUIRED + ['G_output_inventory_exact', 'G_generation_stream_keys_unique', 'G_gate_inventory_exact']
ALL = all(GATES[k] for k in REQUIRED) and all(STATUS.values()); status = ('A10_VALID' if A10_MODE == 'official' else 'SMOKE_PASS') if ALL else 'FAILED'
prov = dict(notebook=f'Step1 Phase A-10 v1.2.5 [{A10_MODE}]', required_gate_count=dict(smoke=len(REQ_A) + 2 + len(REQ_C) + 6 + 3, official=len(REQ_A) + 5 + len(REQ_C) + 10 + 3), status=status, component_status=STATUS, timestamp=time.strftime('%Y-%m-%dT%H:%M:%S'), mode=A10_MODE, config=CFG, master_seed=MASTER_SEED, streams=STREAM, production_path=PROD, threads_live=THREADS_LIVE,
            notebook_identity=dict(basename=NB_BASENAME, head_copy=NB_HEAD, live=NB_LIVE), repo=dict(commit=MT_COMMIT, git_calls=GIT_CALLS), environment=dict(versions=VERS, expected=EXPECTED_VERS, mismatch=VERS_MISMATCH, platform=platform.platform(), in_colab=IN_COLAB),
            assets=ASSET_SHA, module_sha=EXP_MOD, generation_streams=dict(namespaces=GEN_NS, calls=GEN_CALLS), cross_selection_policy=dict(primary='float64 selection (l2-4; A10 supersedes the A8b l2-4 dtype recommendation)', sensitivity='float32 selection (A8b registered path, kept for S2)', TOL_NEAR_TIE=TOL_NEAR_TIE, TOL_EVENTB_MISMATCH=TOL_EVENTB_MISMATCH, DELTA_Q_SENS=DELTA_Q_SENS, W2_full_pool_rms_diagnostic_fraction=0.05, axis_reporting='plane-folded label; raw oriented representative kept (AX, full for calibration and all m runs) and used for float64 T1/T2 evaluation; near-minimizer ambiguity at float32 resolution recorded per flip with plane-folded angle', a8b_TOL_EVAL_unchanged=1e-6,
                                        sandbox_discovery='antipodal flips can change T2 by up to ~1e-3 relative (non-identical B- rows of pixelised antipodal representatives); non-antipodal flips occur at near ties (S+ within float32 resolution); flip rate ~5e-5 per sample; Event B unaffected in 8e4 samples', status='primary float64 selection for l2-4 adopted prospectively in v1.2.4 before any official run; A8b float32 result remains a valid performance benchmark and the sensitivity path; S2 remains float32; per-dtype chunks bound to the A8b CPU winners (float64: 2000, float32: 20000)'), quadrature=DIAG['quadrature_sha256'], event='Event B = {T1 <= T1_obs and T2 <= T2_obs} (A5-frozen primary event); central-band E_sel of v1.0 is superseded', gates=GATES, required_gates=REQUIRED, diagnostic_gates=sorted(set(GATES) - set(REQUIRED)), gate_policy='status is exact over required_gates only; diagnostic gates (e.g. G_m100_usable when m=10 is adopted; stability gates in smoke) are recorded and may be False without affecting status',
            covariance=REC['covariance'], calibration=REC['calibration'], m_sensitivity=dict(runs=MS, decision=M_DECISION), calibration_pathway=CAL, w2=REC['w2'], outputs=outputs, superseded=dict(v1_0='central-band event + float64 selection + chunk 10000; Q~1.08-1.11, m=100, FWFSR 0/200 withdrawn', v1_1='never run officially: float32 product order differed from the A8b child; m fallback unreachable; zero-numerator bootstrap dropped; W2 observed/null mechanism mismatch', v1_2='never run officially: m fallback still blocked by unconditional consistency/finite gates; W2 checkpoint binding without source/whitening; negative-control validity ungated', v1_2_1='never run officially: scan() lacked the A8b non-finite fail-fast; positive-control inventory incomplete', v1_2_2='never run officially: cross-selection equivalence not gated on the m runs and 1e-6 bound inconsistent with the observed antipodal T2 difference; calibration and pseudo shared one random stream', v1_2_3='never run officially: float32 primary with Event-B-only cross-selection gate left W2/pseudo-threshold sensitivity unverified; old exact-antipode probe contradicted the policy', v1_2_4='never run officially: calibration sensitivity gate compared float64 with itself; m-run helper arguments and evidence labels reversed; W2 full-pool RMS misused as a subsample decision bound; provenance kept float32-primary wording'))
atomic_json(prov, os.path.join(OUT, 'a10_provenance.json')); print('STATUS =', status, '| required gates', sum(GATES[k] for k in REQUIRED), '/', len(REQUIRED), '(official would require', prov['required_gate_count']['official'], ') | components', STATUS, '| provenance SHA', sha(os.path.join(OUT, 'a10_provenance.json'))[:16])
assert status != 'FAILED', {k: GATES[k] for k in REQUIRED if not GATES[k]}


## 実行手順
0. v1.2.5 を commit・push（official は live notebook source と origin/main の一致を hard gate する）。
1. **smoke**（fresh runtime・数分）：先頭に `A10_MODE='smoke'` セルを追加 → Run all → `STATUS = SMOKE_PASS`（4 component 全 True）。smoke では版・thread・live-source・m policy/decision gate は記録のみ，official で required（required gate 数は provenance の `required_gate_count` に smoke/official 別に記録）。
2. **official**（fresh runtime・CPU で可・1〜1.5 時間；W₂ null が支配的）：純正 v1.2.5（追加セルなし）を Run all → `STATUS = A10_VALID`。m 判定が unresolved なら A10b 末尾で停止する（fail-fast）。official では期待版（Python 3.13.15・NumPy 2.1.3・SciPy 1.16.3・healpy 1.20.0・POT 0.9.7.post1）との不一致で停止する。
3. 返送：`runs_step1_phaseA/a10_v1.2.5_{smoke,official}/a10_provenance.json` と `checkpoints/`（a10a〜a10d の npz/json）・全セル出力。**OUT は fresh directory**（stale checkpoint は binding 不一致で無視されるが，output inventory は固定名のみを hash する）。
